8.1 Сборка простой ракеты в исходном коде

In [1]:
abstract type Tank end

In [2]:
mutable struct SmallTank <: Tank
    propellant::Float64
end

In [3]:
mutable struct MediumTank <: Tank
    propellant::Float64
end

In [4]:
mutable struct LargeTank <: Tank
    propellant::Float64
end

In [5]:
drymass(::SmallTank) = 40.0
drymass(::MediumTank) = 250.0
drymass(::LargeTank) = 950.0

drymass (generic function with 3 methods)


In [6]:
totalmass(::SmallTank) = 410.0
totalmass(::MediumTank) = 2300.0
totalmass(::LargeTank) = 10200.0

totalmass (generic function with 3 methods)


In [7]:
mutable struct FlexiTank <: Tank
    drymass::Float64
    totalmass::Float64
    propellant::Float64
end

In [8]:
drymass(tank::FlexiTank) = tank.drymass
totalmass(tank::FlexiTank) = tank.totalmass

totalmass (generic function with 4 methods)


In [9]:
propellant(tank::Tank) = tank.propellant

propellant (generic function with 1 method)


In [10]:
function propellant!(tank::Tank, amount::Real)
    tank.propellant = amount
end

propellant! (generic function with 1 method)


In [11]:
isempty(tank::Tank) = tank.propellant <= 0

isempty (generic function with 1 method)


In [12]:
mass(tank::Tank) = drymass(tank) + propellant(tank)

mass (generic function with 1 method)


In [13]:
function refill!(tank::Tank)
    propellant!(tank, totalmass(tank) - drymass(tank))
    tank
end

refill! (generic function with 1 method)


In [14]:
function consume!(tank::Tank, amount::Real)
    remaining = max(propellant(tank) - amount, 0)
    propellant!(tank, remaining)
    remaining
end

consume! (generic function with 1 method)


In [15]:
small = SmallTank(50)

SmallTank(50.0)


In [16]:
consume!(small, 10)

40.0


In [17]:
consume!(small, 10)

30.0


In [18]:
small

SmallTank(30.0)


In [19]:
refill!(small)

SmallTank(370.0)


In [20]:
flexi = FlexiTank(5, 50, 0)

FlexiTank(5.0, 50.0, 0.0)


In [21]:
refill!(flexi)

FlexiTank(5.0, 50.0, 45.0)


8.2 Поддержание инвариантов в исходном коде

In [22]:
function propellant!(tank::Tank, amount::Real)
    if 0 <= amount + drymass(tank) <= totalmass(tank)
        tank.propellant = amount
    else
        msg = "Propellant mass plus dry mass must be less than total mass"
        throw(DomainError(amount, msg))
    end
end

propellant! (generic function with 1 method)


In [23]:
t = FlexiTank(5, 50, 90)

FlexiTank(5.0, 50.0, 90.0)


In [24]:
mass(t), totalmass(t)

(95.0, 50.0)


8.3 Создание объектов с помощью функций-конструкторов

In [25]:
function FlexiTank(drymass::Number, totalmass::Number)
    FlexiTank(drymass, totalmass, totalmass - drymass)
end

FlexiTank


In [26]:
MediumTank() = refill!(MediumTank(0))

MediumTank


In [27]:
LargeTank() = refill!(LargeTank(0))

LargeTank


In [28]:
methods(FlexiTank)

# 3 methods for type constructor:
 [1] FlexiTank(drymass::Float64, totalmass::Float64, propellant::Float64)
     @ none:3
 [2] FlexiTank(drymass, totalmass, propellant)
     @ none:3
 [3] FlexiTank(drymass::Number, totalmass::Number)
     @ none:2


In [29]:
FlexiTank(5, 50)

FlexiTank(5.0, 50.0, 45.0)


In [30]:
MediumTank()

MediumTank(2050.0)


In [31]:
LargeTank()

LargeTank(9250.0)


8.4 Различия между внешними и внутренними конструкторами

In [32]:
mutable struct FlexiTank <: Tank
    drymass::Float64
    totalmass::Float64
    propellant::Float64
    function FlexiTank(drymass::Number, totalmass::Number)
        new(drymass, totalmass, totalmass - drymass)
    end
end

In [33]:
t = FlexiTank(5, 50)

FlexiTank(5.0, 50.0, 45.0)


In [34]:
methods(FlexiTank)

# 3 methods for type constructor:
 [1] FlexiTank(drymass::Float64, totalmass::Float64, propellant::Float64)
     @ none:3
 [2] FlexiTank(drymass, totalmass, propellant)
     @ none:3
 [3] FlexiTank(drymass::Number, totalmass::Number)
     @ none:6


8.5 Моделирование ракетных двигателей и полезной нагрузки

In [35]:
struct Payload
    mass::Float64
end

In [36]:
abstract type Engine end

In [37]:
struct CustomEngine <: Engine
    mass::Float64
    thrust::Float64
    Isp::Float64
end

In [38]:
mass(engine::CustomEngine) = engine.mass
thrust(engine::CustomEngine) = engine.thrust
Isp(engine::CustomEngine) = engine.Isp

Isp (generic function with 1 method)


In [39]:
struct Rutherford <: Engine end
struct Merlin <: Engine end

In [40]:
mass(::Rutherford) = 35.0
thrust(::Rutherford) = 25000.0
Isp(::Rutherford) = 311.0

Isp (generic function with 2 methods)


In [41]:
mass(::Merlin) = 470.0
thrust(::Merlin) = 845e3
Isp(::Merlin) = 282.0

Isp (generic function with 3 methods)


In [42]:
rutherford = Rutherford()

Rutherford()


In [43]:
mass(rutherford)

35.0


In [44]:
thrust(rutherford)

25000.0


In [45]:
Isp(rutherford)

311.0


In [46]:
merlin = Merlin()

Merlin()


In [47]:
mass(merlin)

470.0


In [48]:
thrust(merlin)

845000.0


In [49]:
Isp(merlin)

282.0


In [50]:
g0 = 9.80665
mass_flow(thrust, Isp) = thrust / (Isp * g0)

mass_flow (generic function with 1 method)


In [51]:
mass_flow(thrust(rutherford), Isp(rutherford))

8.197075667025148


8.6 Сборка простой ракеты

In [52]:
struct Rocket
    payload::Payload
    tank::Tank
    engine::Engine
end

In [53]:
SmallTank() = refill!(SmallTank(0))

SmallTank


In [54]:
engine = Rutherford()

Rutherford()


In [55]:
tank = SmallTank()

SmallTank(370.0)


In [56]:
payload = Payload(300)

Payload(300.0)


In [57]:
rocket = Rocket(payload, tank, engine)

Rocket(Payload(300.0), SmallTank(370.0), Rutherford())


8.7 Создание ракеты с несколькими ступенями и двигателями

In [58]:
abstract type Rocket end

In [59]:
struct Payload <: Rocket
    mass::Float64
end

In [60]:
struct StagedRocket <: Rocket
    nextstage::Rocket
    tank::Tank
    engine::Engine
end

In [61]:
function Rocket(payload::Rocket, tank::Tank, engine::Engine)
    StagedRocket(payload, tank, engine)
end

Rocket


In [62]:
thrust(r::Payload) = 0.0

thrust (generic function with 4 methods)


In [63]:
thrust(r::StagedRocket) = thrust(r.engine)

thrust (generic function with 5 methods)


In [64]:
struct Cluster <: Engine
    engine::Engine
    count::Int
end

In [65]:
Isp(cl::Cluster) = Isp(cl.engine)

Isp (generic function with 4 methods)


In [66]:
mass(cl::Cluster) = mass(cl.engine) * cl.count

mass (generic function with 5 methods)


In [67]:
thrust(cl::Cluster) = thrust(cl.engine) * cl.count

thrust (generic function with 6 methods)


In [68]:
mass(payload::Payload) = payload.mass

mass (generic function with 6 methods)


In [69]:
function mass(r::StagedRocket)
    mass(r.nextstage) + mass(r.tank) + mass(r.engine)
end

mass (generic function with 7 methods)


In [70]:
function update!(r::StagedRocket, t::Number, Δt::Number)
    mflow = mass_flow(thrust(r), Isp(r.engine))
    consume!(r.tank, mflow * Δt)
end

update! (generic function with 1 method)


In [71]:
update!(r::Payload, t::Number, Δt::Number) = nothing

update! (generic function with 2 methods)


In [72]:
struct Curie <: Engine end
mass(::Curie) = 8.0
thrust(::Curie) = 120.0
Isp(::Curie) = 317.0

Isp (generic function with 5 methods)


In [73]:
payload = Payload(300)

Payload(300.0)


In [74]:
thirdstage = Rocket(payload, SmallTank(), Curie())

StagedRocket(Payload(300.0), SmallTank(370.0), Curie())


In [75]:
secondstage = Rocket(thirdstage, MediumTank(), Rutherford())

StagedRocket(StagedRocket(Payload(300.0), SmallTank(370.0), Curie()), MediumTank(2050.0), Rutherford())


In [76]:
booster = Rocket(secondstage, LargeTank(), Cluster(Rutherford(), 9))

StagedRocket(StagedRocket(StagedRocket(Payload(300.0), SmallTank(370.0), Curie()), MediumTank(2050.0), Rutherford()), LargeTank(9250.0), Cluster(Rutherford(), 9))


In [77]:
mass(payload)

300.0


In [78]:
mass(thirdstage)

718.0


In [79]:
mass(secondstage)

3053.0


In [80]:
mass(booster)

13568.0


In [81]:
thrust(booster)

225000.0


8.8 Запуск ракеты в космос

In [82]:
function launch!(rocket::Rocket, Δt::Real)
    g = 9.80665 # вызываемое гравитацией ускорение
    v = 0.0     # скорость
    h = 0.0     # высота
    for t in 0:Δt:1000
        m = mass(rocket)
        F = thrust(rocket) - m*g
        remaining = update!(rocket, t, Δt)
        if remaining == 0 || F <= 0
            return (t, h)
        end
        h += v*Δt
        a = F/m
        v += a*Δt
    end
end

launch! (generic function with 1 method)


In [83]:
engine = Rutherford()

Rutherford()


In [84]:
tank = SmallTank()

SmallTank(370.0)


In [85]:
payload = Payload(300)

Payload(300.0)


In [86]:
rocket = Rocket(payload, tank, engine)

StagedRocket(Payload(300.0), SmallTank(370.0), Rutherford())


In [87]:
launch!(rocket, 0.5)

(45.0, 31117.8036364191)


In [88]:
secondstage = Rocket(payload, MediumTank(), Rutherford())

StagedRocket(Payload(300.0), MediumTank(2050.0), Rutherford())


In [89]:
booster = Rocket(secondstage, LargeTank(), Cluster(Rutherford(), 9))

StagedRocket(StagedRocket(Payload(300.0), MediumTank(2050.0), Rutherford()), LargeTank(9250.0), Cluster(Rutherford(), 9))


In [90]:
launch!(booster, 0.5)

(125.0, 107408.46096155819)
